# Public data experiments
This is the notebook to run the public data experiment

In [1]:
import sys
from pathlib import Path

# Add the workspace root to sys.path
workspace_root = Path("/Users/eduardohome/workspace/benchmark_credit_scoring")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

# Now you can import from src
from src.utils.csv_config import read_csv_by_name, create_imbalanced_datasets, merge_xai_results
from src.model.functions import save_model, load_model, get_split_data
from src.xai.functions import get_train_iterations_xai

try:
    from src.model.functions import get_train_model
except Exception as exc:
    # provide helpful message if import fails
    print("Failed to import model utilities:", exc)
    print("This often means a missing dependency such as libomp for LightGBM.")
    print("On macOS you can install it with `brew install libomp` and then restart the kernel.")
    raise

# General imports
import pandas as pd

In [2]:
# -------------------------
# Settings
# -------------------------
data_dir = Path("/Users/eduardohome/workspace/benchmark_credit_scoring/data")
datasets_to_run = ["australian", "german", "japan", "kgccdea", "kggmsc", "taiwan", "th02"]
target_col = "target"

# Edit this list whenever you want more/less imbalance datasets
imbalance_levels = [10, 15, 20, 25, 30]

# list of estimators understood by functions.get_model
model_choices = ["RandomForest", "XGBoost", "LightGBM", "BoostDT", "BoostSGB",
                 "BagNN", "AdaBoost-DT", "SVM-Rbf", "ANN"]
product = "default"


In [3]:
for dataset_name in datasets_to_run:
    dataset_path = data_dir / "input" / f"{dataset_name}.csv"
    # Check if file exists and read it using the helper from src.csv_config
    if dataset_path.exists():
        # Use the dataset-specific defaults (german uses tab separator)
        create_imbalanced_datasets(
            dataset_name=dataset_name,
            data_dir=data_dir,
            imbalance_levels=imbalance_levels,  # <-- modify freely
            target_col="target",
            )

Original dataset: australian
Total samples: 690
Positive samples: 307
Negative samples: 383
Positive rate: 0.4449

australian_10.csv | Total=425 | Pos=42 | Rate=0.0988 | keep all negatives (383), downsample positives to 42
australian_15.csv | Total=450 | Pos=67 | Rate=0.1489 | keep all negatives (383), downsample positives to 67
australian_20.csv | Total=478 | Pos=95 | Rate=0.1987 | keep all negatives (383), downsample positives to 95
australian_25.csv | Total=510 | Pos=127 | Rate=0.2490 | keep all negatives (383), downsample positives to 127
australian_30.csv | Total=547 | Pos=164 | Rate=0.2998 | keep all negatives (383), downsample positives to 164

Finished generating imbalanced datasets.
Original dataset: german
Total samples: 1000
Positive samples: 300
Negative samples: 700
Positive rate: 0.3000

german_10.csv | Total=777 | Pos=77 | Rate=0.0991 | keep all negatives (700), downsample positives to 77
german_15.csv | Total=823 | Pos=123 | Rate=0.1495 | keep all negatives (700), downs

In [4]:
results = []

for dataset_name in datasets_to_run:
    # variants for this dataset
    dataset_variants = [dataset_name] + [f"{dataset_name}_{pct}" for pct in imbalance_levels]

    # ensure output dir exists
    out_dir = data_dir / "output" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 90)
    print(f"BASE DATASET: {dataset_name}")
    print("#" * 90)

    for dataset_variant in dataset_variants:
        dataset_path = data_dir / "intermediate" / dataset_name / f"{dataset_variant}.csv"

        if not dataset_path.exists():
            print(f"[SKIP] File not found: {dataset_path}")
            continue

        # Read each dataset variant (uses your resolver logic in csv_config.py)
        df = pd.read_csv(str(dataset_path))

        # Basic safety checks
        if target_col not in df.columns:
            print(f"[SKIP] {dataset_variant}: target column '{target_col}' not found. Columns={list(df.columns)[:10]}...")
            continue

        # Quick dataset stats
        n_total = len(df)
        # Make sure sum/mean work even if target is int-like
        y = pd.to_numeric(df[target_col], errors="coerce")
        n_pos = int((y == 1).sum())
        pos_rate = float((y == 1).mean())

        print("\n" + "=" * 80)
        print(f"DATASET: {dataset_variant} | n={n_total} | pos={n_pos} | pos_rate={pos_rate:.4f}")
        print("=" * 80)

        # Prepare column lists for this df
        feature_cols     = df.columns.tolist()
        numerical_cols   = df.select_dtypes(include="number").columns.tolist()
        categorical_cols = df.select_dtypes(include="object").columns.tolist()
        ids_cols         = []
        target_cols      = [target_col]
        exclude_cols     = []

        # Optionally skip if only one class present (prevents stratify crash)
        n_neg = n_total - n_pos
        if n_pos == 0 or n_neg == 0:
            print(f"[SKIP] {dataset_variant}: only one class present (pos={n_pos}, neg={n_neg}).")
            continue

        for model_choice in model_choices:
            print(f"\n=== training {model_choice} on {dataset_variant} ===")

            pipeline, best_params, best_score = get_train_model(
                data=df,
                feature_cols=feature_cols,
                numerical_cols=numerical_cols,
                categorical_cols=categorical_cols,
                ids_cols=ids_cols,
                target_cols=target_cols,
                exclude_cols=exclude_cols,
                product=product,
                algorithm=model_choice,
                test_mode=True,  # set False for full search
            )

            print("Best hyperparameters:", best_params)
            print("Best CV score:", best_score)

            model_path = out_dir / f"{dataset_variant}_{model_choice}.pkl"
            save_model(str(model_path), pipeline, environment="local")
            print(f"trained pipeline saved to {model_path}")

            results.append(
                {
                    "base_dataset": dataset_name,
                    "dataset_variant": dataset_variant,
                    "model": model_choice,
                    "n_total": n_total,
                    "n_pos": n_pos,
                    "pos_rate": pos_rate,
                    "params": best_params,
                    "score": best_score,
                }
            )

results_df = pd.DataFrame(results)
results_df


##########################################################################################
BASE DATASET: australian
##########################################################################################
[SKIP] File not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/intermediate/australian/australian.csv

DATASET: australian_10 | n=425 | pos=42 | pos_rate=0.0988

=== training RandomForest on australian_10 ===
Search space for algorithm=RandomForest, product=default, test_mode=True: {'model__n_estimators': Integer(low=50, high=300, prior='uniform', transform='identity'), 'model__max_depth': Integer(low=1, high=50, prior='uniform', transform='identity'), 'model__min_samples_split': Integer(low=2, high=20, prior='uniform', transform='identity'), 'model__min_samples_leaf': Integer(low=1, high=10, prior='uniform', transform='identity'), 'model__max_features': Categorical(categories=('sqrt',), prior=None), 'model__bootstrap': Categorical(categories=(True, False), prior

,base_dataset,dataset_variant,model,n_total,n_pos,pos_rate,params,score
0,australian,australian_10,RandomForest,425,42,0.098824,"{'model__bootstrap': False, 'model__class_weig...",0.753858
1,australian,australian_10,XGBoost,425,42,0.098824,"{'model__booster': 'gbtree', 'model__colsample...",0.097633
2,australian,australian_10,LightGBM,425,42,0.098824,{'model__bagging_fraction': 0.9293468848796316...,0.097633
3,australian,australian_10,BoostDT,425,42,0.098824,"{'model__learning_rate': 0.07222603429369405, ...",0.694188
4,australian,australian_10,BoostSGB,425,42,0.098824,"{'model__learning_rate': 0.07222603429369405, ...",0.723732
...,...,...,...,...,...,...,...,...
310,th02,th02_30,BoostSGB,1076,323,0.300186,"{'model__learning_rate': 0.07222603429369405, ...",0.428548
311,th02,th02_30,BagNN,1076,323,0.300186,"{'model__estimator__activation': 'relu', 'mode...",0.432200
312,th02,th02_30,AdaBoost-DT,1076,323,0.300186,"{'model__estimator__max_depth': 3, 'model__est...",0.358007
313,th02,th02_30,SVM-Rbf,1076,323,0.300186,"{'model__C': 5.216600029793867, 'model__gamma'...",0.374989


In [5]:
# -------------------------
# Loop XAI iterations
# -------------------------
for dataset_name in datasets_to_run:

    dataset_variants = [dataset_name] + [f"{dataset_name}_{lvl}" for lvl in imbalance_levels]

    out_dir = data_dir / "output" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 90)
    print(f"BASE DATASET: {dataset_name}")
    print("#" * 90)

    for dataset_variant in dataset_variants:

        dataset_path = data_dir / "intermediate" / dataset_name / f"{dataset_variant}.csv"

        if not dataset_path.exists():
            print(f"[SKIP] File not found: {dataset_path}")
            continue

        # Same reader used during training
        path = str(dataset_path)
        df = pd.read_csv(str(path))

        if target_col not in df.columns:
            print(f"[SKIP] {dataset_variant}: target column '{target_col}' not found.")
            continue

        n_total = len(df)
        y = pd.to_numeric(df[target_col], errors="coerce")
        n_pos = int((y == 1).sum())
        pos_rate = float((y == 1).mean())

        print("\n" + "=" * 80)
        print(f"DATASET: {dataset_variant} | n={n_total} | pos={n_pos} | pos_rate={pos_rate:.4f}")
        print("=" * 80)

        n_neg = n_total - n_pos
        if n_pos == 0 or n_neg == 0:
            print(f"[SKIP] {dataset_variant}: only one class present.")
            continue

        # Same column preparation as training
        feature_cols     = df.columns.tolist()
        numerical_cols   = df.select_dtypes(include="number").columns.tolist()
        categorical_cols = df.select_dtypes(include="object").columns.tolist()
        ids_cols         = []
        target_cols      = [target_col]
        exclude_cols     = []

        # Data split
        x_train, x_test, y_train, y_test, numerical, categorical = get_split_data(
            data=df,
            feature_cols=feature_cols,
            numerical_cols=numerical_cols,
            categorical_cols=categorical_cols,
            ids_cols=ids_cols,
            target_cols=target_cols,
            exclude_cols=exclude_cols,
            random_state=123,
        )

        for model_choice in model_choices:

            model_path = out_dir / f"{dataset_variant}_{model_choice}.pkl"

            if not model_path.exists():
                print(f"[SKIP] Model not found: {model_path}")
                continue

            print(f"Running XAI iterations for {dataset_variant} | {model_choice}")

            pipeline = load_model(str(model_path))

            _, _, _, _, _, df_resume = get_train_iterations_xai(
                pipeline,
                x_train,
                x_test,
                y_train,
                n_iterations=5,
                percentage=90,
            )

            out_file = out_dir / f"{dataset_variant}_{model_choice}_xai_iterations.csv"

            df_resume.to_csv(out_file, index=False)

            print(f"[OK] Saved: {out_file}")


##########################################################################################
BASE DATASET: australian
##########################################################################################
[SKIP] File not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/intermediate/australian/australian.csv

DATASET: australian_10 | n=425 | pos=42 | pos_rate=0.0988
Running XAI iterations for australian_10 | RandomForest
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/australian_10_RandomForest.pkl
n_iteration  rankstab_global  feature  feature_importance  value_stab  \
0                        1.2        0            0.530726    0.209499   
1                        0.7        1           12.435945    0.215324   
2                        0.2        2            2.691430    0.176353   
3                        2.8        3            0.658626    0.730957   
4                        0.2        4    

ExactExplainer explainer: 11it [03:11, 19.15s/it]                        
ExactExplainer explainer: 11it [03:00, 18.02s/it]                        
ExactExplainer explainer: 11it [03:13, 19.35s/it]                        
ExactExplainer explainer: 11it [03:07, 18.74s/it]                        
ExactExplainer explainer: 11it [03:13, 19.32s/it]                        


n_iteration  rankstab_global  feature  feature_importance  value_stab  \
0                        1.2        0            9.447918    0.450543   
1                        1.3        1            2.655385    0.280651   
2                        1.7        2           12.537257    0.603890   
3                        5.0        3            2.286585    0.704288   
4                        1.7        4            3.059922    0.252473   
5                        1.5        5            3.169808    0.409258   
6                        0.0        6           41.426310    0.480222   
7                        3.5        7            2.681276    0.519296   
8                        0.7        8           17.425012    0.530679   
9                        6.8        9            5.310526    0.574762   

n_iteration  rankstab_global_weight  value_stab_weight  
0                         11.337502           4.256694  
1                          3.452000           0.745238  
2                        

ExactExplainer explainer: 11it [02:30, 15.05s/it]                        
ExactExplainer explainer: 11it [02:32, 15.29s/it]                        
ExactExplainer explainer: 11it [05:04, 30.45s/it]                        
ExactExplainer explainer: 11it [02:34, 15.45s/it]                        
ExactExplainer explainer: 11it [02:26, 14.68s/it]                        


n_iteration  rankstab_global  feature  feature_importance  value_stab  \
0                        0.2        0           13.847970    0.218850   
1                        1.2        1            4.062313    0.334399   
2                        0.2        2           23.252289    0.204634   
3                        8.7        3            5.660395    0.758475   
4                        2.7        4            3.906528    0.256767   
5                        5.8        5            3.427770    0.405119   
6                        0.2        6           30.801501    0.244116   
7                        1.7        7            2.413008    0.287815   
8                        1.3        8            9.561662    0.439924   
9                        1.8        9            3.066564    0.520955   

n_iteration  rankstab_global_weight  value_stab_weight  
0                          2.769594           3.030632  
1                          4.874775           1.358433  
2                        

...

n_iteration  rankstab_global  feature  feature_importance  value_stab  \
0                     22.300        0            1.570737    0.317324   
1                      6.250        1            1.469226    0.330150   
2                     16.550        2            1.376621    0.480739   
3                     15.700        3            1.291138    0.376983   
4                      1.300        4            1.127297    0.216652   
5                      0.000        5           52.297336    0.195433   
6                      1.200        6            6.038966    0.287530   
7                      0.200        7            3.973144    0.246676   
8                      0.800        8            2.972289    0.121550   
9                      2.300        9            6.637342    0.584832   
10                     1.200       10            6.665836    0.225874   
11                     0.800       11            2.048013    0.213287   
12                    11.625       12            1.

PermutationExplainer explainer: 24it [04:28, 12.21s/it]                        
PermutationExplainer explainer: 24it [05:23, 14.08s/it]                        
PermutationExplainer explainer: 24it [04:03, 10.60s/it]                        
PermutationExplainer explainer: 24it [04:05, 11.17s/it]                        
PermutationExplainer explainer: 24it [05:37, 14.65s/it]                        


n_iteration  rankstab_global  feature  feature_importance  value_stab  \
0                        5.3        0            3.159981    0.090553   
1                        6.3        1            2.367071    0.344895   
2                        0.8        2            5.972269    0.113661   
3                        0.2        3            0.944496    0.156946   
4                        3.8        4            1.519038    0.271928   
5                        0.0        5           29.990672    0.103574   
6                        0.0        6           10.674610    0.118563   
7                       10.3        7            3.176942    0.234256   
8                        4.7        8            4.309065    0.179771   
9                        7.7        9            4.801984    0.227886   
10                       9.2       10            3.464213    0.338007   
11                      10.3       11            4.083277    0.258214   
12                      12.5       12            2.

PermutationExplainer explainer: 24it [05:06, 13.31s/it]                        
PermutationExplainer explainer: 24it [03:47, 10.35s/it]                        
PermutationExplainer explainer: 24it [03:48, 10.40s/it]                        
PermutationExplainer explainer: 24it [20:03, 54.70s/it]                         
PermutationExplainer explainer: 24it [03:45, 10.23s/it]                        


n_iteration  rankstab_global  feature  feature_importance  value_stab  \
0                        9.5        0            3.044005    0.202840   
1                       15.3        1            2.482842    0.224033   
2                        0.5        2            1.890728    0.081112   
3                       15.5        3            1.243440    0.352966   
4                        8.0        4            1.651191    0.187706   
5                        0.0        5           43.048042    0.053625   
6                        0.0        6           11.314762    0.131202   
7                        2.7        7            4.753061    0.358130   
8                       39.8        8            2.018571    0.363677   
9                        4.3        9            2.333468    0.220521   
10                       0.8       10            1.758848    0.186407   
11                      43.0       11            2.081819    0.468123   
12                       1.0       12            3.

In [3]:
merge_xai_results(
    data_dir=data_dir,
    datasets_to_run=datasets_to_run,
    imbalance_levels=imbalance_levels,
    model_choices=model_choices,
)


Processing dataset: australian
Saved merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/australian_xai_all_models.csv

Processing dataset: german
Saved merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/german/german_xai_all_models.csv

Processing dataset: japan
Saved merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/japan/japan_xai_all_models.csv

Processing dataset: kgccdea
Saved merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/kgccdea/kgccdea_xai_all_models.csv

Processing dataset: kggmsc
Saved merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/kggmsc/kggmsc_xai_all_models.csv

Processing dataset: taiwan
Saved merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_xai_all_models.csv

Processing dataset: th02
Saved merged dataset: /Users/eduardohome/workspace/benchmark_credit_

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

g = sns.catplot(
    data=df,
    x="imbalance",
    y="stability",
    row="model",
    kind="box",
    sharey=True,
    height=1.6,
    aspect=4,
    fliersize=2,
)

# Set common axis labels
g.set_axis_labels("Default rate", "Global stability")

# Fix y-axis range
for ax in g.axes.flatten():
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'seaborn'